#Transformers:<br>


*   <small>Transformers train on provided data and predict the next word based on the data trained.<br>
P(next token | previous words)
*   It is basically a massive probability engine trained to continue sequences in intelligent ways.</small>



<br>

Data Processing

In [ ]:
!pip install transformers datasets bertviz umap-learn

In [ ]:
import pandas as pd
from datasets import load_dataset

dataset = load_dataset("emotion")
dataset.set_format(type="pandas")

In [ ]:
df=dataset['train'][:]
df.head()

In [ ]:
classes = dataset['train'].features['label'].names #Writing a function for displaying label names
classes

In [ ]:
df['label_name'] = df['label'].apply(lambda x: classes[x])
df.head()

<br>

Data Analysis

In [ ]:
import matplotlib.pyplot as plt

label_counts = df['label_name'].value_counts(ascending=False)
label_counts.plot(kind='bar')
plt.title('Frequency of labels')
plt.show()

<br>

Data Preprocessing (+ Tokenization)

In [ ]:
from transformers import AutoTokenizer

model_ckpt = "justin871030/bert-base-uncased-goemotions-group-finetuned"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

In [ ]:
#Attributes of 🤗 transformer tokenizer
tokenizer.vocab_size, tokenizer.model_max_length

In [ ]:
!pip install torch

In [ ]:
dataset

<small>Tokenizing:</small>

In [ ]:
dataset.reset_format()

In [ ]:
def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True, max_length=128)

dataset_encoded = dataset.map(tokenize, batched=True)

<small>By putting labels in the same dictionary, we can pass the whole batch to the model in one call. Hugging Face models recognize labels in the input dictionary automatically and compute the loss internally. </small>

<br>

Load Transformer Model:

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=6, ignore_mismatched_sizes=True)

<br>

Trainer = (Loss computation, Optimizer, Learning rate scheduler, Data loader, Collate function, Batching, Shuffling)

In [ ]:
from transformers import Trainer, TrainingArguments

batch_size = 64
model_name = "bert-goemotion-finetuned-emotion-recog"
training_args = TrainingArguments(
    output_dir=model_name,
    num_train_epochs=5,
    learning_rate=3e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    eval_strategy='epoch',
    warmup_steps=500,
    lr_scheduler_type='cosine',
    save_strategy='epoch',
)


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
  labels = pred.label_ids
  preds = pred.predictions.argmax(-1)
  f1 = f1_score(labels, preds, average='weighted')
  acc = accuracy_score(labels, preds)
  return {'accuracy': acc, 'f1': f1}

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=dataset_encoded['train'],
    eval_dataset=dataset_encoded['validation'],
    processing_class=tokenizer
)

<br>

Training

In [ ]:
trainer.train()

<br>

Testing

In [ ]:
preds_outputs = trainer.predict(dataset_encoded['test'])
preds_outputs.metrics

In [ ]:
import numpy as np
y_preds = np.argmax(preds_outputs.predictions, axis=1) #preds_outputs: model is thinking, y_preds: model makes a choice
y_true = dataset_encoded['test'][:]['label'] #y_true: actual correct answer

In [ ]:
from sklearn.metrics import classification_report
print(classes)
print(classification_report(y_true, y_preds, target_names=classes))

<br>

Prediction

In [ ]:
import torch

text = "If I try harder, I know I will get better."
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoded_text = tokenizer(text, return_tensors='pt').to(device)
with torch.no_grad():
  output = model(**encoded_text)

logits = output.logits
print(logits)
preds = torch.argmax(logits, dim=1)
print(preds, classes[preds])